In [2]:
!pip install waymo-open-dataset-tf-2-12-0==1.6.7 --no-deps -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 36.3 MB/s eta 0:00:00


In [3]:
import tensorflow as tf
from waymo_open_dataset.protos import scenario_pb2
print("TF version:", tf.__version__)
print("Scenario proto loaded OK")

TF version: 2.20.0
Scenario proto loaded OK


In [4]:
from google.colab import auth
auth.authenticate_user()

In [5]:
!gcloud storage cp gs://waymo_open_dataset_motion_v_1_3_1/uncompressed/scenario/validation_interactive/validation_interactive.tfrecord-00000-of-00150 /content/

Copying gs://waymo_open_dataset_motion_v_1_3_1/uncompressed/scenario/validation_interactive/validation_interactive.tfrecord-00000-of-00150 to file:///content/validation_interactive.tfrecord-00000-of-00150

Average throughput: 215.3MiB/s


To take a quick anonymous survey, run:
  $ gcloud survey



In [6]:
!ls -lh /content/

total 253M
drwxr-xr-x 1 root root 4.0K Apr 16 13:28 sample_data
-rw-r--r-- 1 root root 253M May  6 22:27 validation_interactive.tfrecord-00000-of-00150


In [7]:
import tensorflow as tf
from waymo_open_dataset.protos import scenario_pb2
import pandas as pd
import os

INPUT_FILE = '/content/validation_interactive.tfrecord-00000-of-00150'
OUTPUT_DIR = '/content/processed'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Enum lookups so the parquet has human-readable strings rather than ints
OBJECT_TYPES = {0: 'unset', 1: 'vehicle', 2: 'pedestrian', 3: 'cyclist', 4: 'other'}
LANE_TYPES   = {0: 'undefined', 1: 'freeway', 2: 'surface_street', 3: 'bike_lane'}

scenes, agents, agent_states = [], [], []
map_lanes, map_crosswalks, map_road_edges = [], [], []

dataset = tf.data.TFRecordDataset([INPUT_FILE], compression_type='')

for i, raw in enumerate(dataset):
    scenario = scenario_pb2.Scenario()
    scenario.ParseFromString(raw.numpy())

    scene_id = scenario.scenario_id
    sdc_idx = scenario.sdc_track_index
    objects_of_interest = set(scenario.objects_of_interest)

    scenes.append({
        'scene_id': scene_id,
        'n_timesteps': len(scenario.timestamps_seconds),
        'current_time_index': scenario.current_time_index,
        'sdc_track_index': sdc_idx,
        'n_tracks': len(scenario.tracks),
        'n_map_features': len(scenario.map_features),
    })

    # Agents + per-timestep states
    for track_idx, track in enumerate(scenario.tracks):
        agents.append({
            'scene_id': scene_id,
            'track_id': track.id,
            'track_index': track_idx,
            'object_type': OBJECT_TYPES.get(track.object_type, 'unknown'),
            'is_sdc': track_idx == sdc_idx,
            'is_object_of_interest': track_idx in objects_of_interest,
        })
        for t, state in enumerate(track.states):
            agent_states.append({
                'scene_id': scene_id,
                'track_id': track.id,
                'timestep': t,
                'time_seconds': scenario.timestamps_seconds[t],
                'center_x': state.center_x,
                'center_y': state.center_y,
                'center_z': state.center_z,
                'length': state.length,
                'width': state.width,
                'height': state.height,
                'heading': state.heading,
                'velocity_x': state.velocity_x,
                'velocity_y': state.velocity_y,
                'valid': state.valid,
            })

    # Map features — split by feature type into separate tables
    for feature in scenario.map_features:
        if feature.HasField('lane'):
            lane = feature.lane
            for pt_idx, pt in enumerate(lane.polyline):
                map_lanes.append({
                    'scene_id': scene_id,
                    'feature_id': feature.id,
                    'lane_type': LANE_TYPES.get(lane.type, 'unknown'),
                    'speed_limit_mph': lane.speed_limit_mph,
                    'point_idx': pt_idx,
                    'x': pt.x, 'y': pt.y, 'z': pt.z,
                })
        elif feature.HasField('crosswalk'):
            for pt_idx, pt in enumerate(feature.crosswalk.polygon):
                map_crosswalks.append({
                    'scene_id': scene_id,
                    'feature_id': feature.id,
                    'point_idx': pt_idx,
                    'x': pt.x, 'y': pt.y, 'z': pt.z,
                })
        elif feature.HasField('road_edge'):
            for pt_idx, pt in enumerate(feature.road_edge.polyline):
                map_road_edges.append({
                    'scene_id': scene_id,
                    'feature_id': feature.id,
                    'edge_type': int(feature.road_edge.type),
                    'point_idx': pt_idx,
                    'x': pt.x, 'y': pt.y, 'z': pt.z,
                })

    if (i + 1) % 25 == 0:
        print(f"Processed {i+1} scenes...")

print(f"\nDone. Totals:")
print(f"  scenes: {len(scenes)}")
print(f"  agents: {len(agents)}")
print(f"  agent_states: {len(agent_states):,}")
print(f"  lane points: {len(map_lanes):,}")
print(f"  crosswalk points: {len(map_crosswalks):,}")
print(f"  road edge points: {len(map_road_edges):,}")

Processed 25 scenes...
Processed 50 scenes...
Processed 75 scenes...
Processed 100 scenes...
Processed 125 scenes...
Processed 150 scenes...
Processed 175 scenes...
Processed 200 scenes...
Processed 225 scenes...
Processed 250 scenes...
Processed 275 scenes...

Done. Totals:
  scenes: 282
  agents: 19021
  agent_states: 1,730,911
  lane points: 3,385,050
  crosswalk points: 9,474
  road edge points: 1,654,334


In [8]:
pd.DataFrame(scenes).to_parquet(f'{OUTPUT_DIR}/scenes.parquet', index=False)
pd.DataFrame(agents).to_parquet(f'{OUTPUT_DIR}/agents.parquet', index=False)
pd.DataFrame(agent_states).to_parquet(f'{OUTPUT_DIR}/agent_states.parquet', index=False)
pd.DataFrame(map_lanes).to_parquet(f'{OUTPUT_DIR}/map_lanes.parquet', index=False)
pd.DataFrame(map_crosswalks).to_parquet(f'{OUTPUT_DIR}/map_crosswalks.parquet', index=False)
pd.DataFrame(map_road_edges).to_parquet(f'{OUTPUT_DIR}/map_road_edges.parquet', index=False)

!ls -lh /content/processed/

import shutil
from google.colab import files
shutil.make_archive('/content/processed', 'zip', '/content/processed')
files.download('/content/processed.zip')

total 134M
-rw-r--r-- 1 root root  65K May  6 22:30 agents.parquet
-rw-r--r-- 1 root root  31M May  6 22:30 agent_states.parquet
-rw-r--r-- 1 root root 223K May  6 22:31 map_crosswalks.parquet
-rw-r--r-- 1 root root  70M May  6 22:31 map_lanes.parquet
-rw-r--r-- 1 root root  34M May  6 22:31 map_road_edges.parquet
-rw-r--r-- 1 root root  12K May  6 22:30 scenes.parquet


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>